## Homework 8: GPUs

## Due Date: April 24, 2026, 6:00pm



#### Firstname Lastname: Dayne Lee

#### E-mail: dl5635@nyu.edu

#### Enter your solutions and submit this notebook

---

**Problem 1 (100p)**


Write two programs which will be able to run in parallel on a GPU, one using Numba/CUDA (50p), one using PyOpenCL (50p).


Each program will:

- draw two random vectors $\vec u$ and $\vec v$ from $[0,1]^N$ where $N = 10^7$;


- calculate and output similarity between $\vec u$ and $\vec v$.




The similarity between two vectors $\vec u$ and $\vec v$ is defined here as a `cosine` value of the angle between them $\measuredangle \left( \vec u, \vec v \right)$. That is, the program returns: 

$$\cos \left( \measuredangle \left( \vec u, \vec v \right) \right).$$


Note that the output is a real value and must belong to $[-1, 1]$.

In [1]:
import os, ctypes.util
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("libcuda location    =", ctypes.util.find_library("cuda"))

from numba import cuda
print(cuda.detect())

CUDA_VISIBLE_DEVICES = 0
libcuda location    = libcuda.so.1


Found 1 CUDA devices
id 0    b'Tesla V100-SXM2-16GB'                              [SUPPORTED]
                      Compute Capability: 7.0
                           PCI Device ID: 0
                              PCI Bus ID: 98
                                    UUID: GPU-9b742555-f188-af1f-5d10-785482dda2ff
                                Watchdog: Disabled
             FP32/FP64 Performance Ratio: 2
Summary:
	1/1 devices are supported
True


In [2]:
# %env NUMBA_ENABLE_CUDASIM=1

In [3]:
import math

from numba import cuda, float32
import numpy as np
import pyopencl as cl

np.random.seed(42)

In [4]:
# data: random vectors (on CPU)

N = 10**7
u = np.random.rand(N).astype(np.float32)
v = np.random.rand(N).astype(np.float32)


In [5]:
# launch config for CUDA kernels
block_dim = 1024
grid_dim = int(N / block_dim) + 1


### Naive Numba/CUDA version

In [6]:
# @cuda.jit
# def cos_sim_kernel(u, v, result):
#     idx = cuda.grid(1)

#     if idx >= u.size:
#         return

#     # each thread computes one element
#     dot = u[idx] * v[idx]
#     u_sq = u[idx] * u[idx]
#     v_sq = v[idx] * v[idx]

#     # atomic add (accumulate results)
#     cuda.atomic.add(result, 0, dot)
#     cuda.atomic.add(result, 1, u_sq)
#     cuda.atomic.add(result, 2, v_sq)

In [7]:
# result = np.zeros(3, dtype=np.float32)  # [dot, u_norm, v_norm]

# u_d = cuda.to_device(u)
# v_d = cuda.to_device(v)
# result_d = cuda.to_device(result)

# cos_sim_kernel[grid_dim, block_dim](u_d, v_d, result_d)
# cuda.synchronize()
# result = result_d.copy_to_host()

# dot, u_norm, v_norm = result
# cos_sim = dot / (np.sqrt(u_norm) * np.sqrt(v_norm))

# print(f"Cosine similarity: {cos_sim}")

### Shared memory Numba/CUDA version

In [8]:
@cuda.jit
def cos_sim_kernel_sm(u, v, result):
    # shared memory per block
    sm_dot = cuda.shared.array(1024, dtype=float32)
    sm_u = cuda.shared.array(1024, dtype=float32)
    sm_v = cuda.shared.array(1024, dtype=float32)

    tid = cuda.threadIdx.x
    idx = cuda.grid(1)

    # Step 1: compute local values
    if idx < u.size:
        sm_dot[tid] = u[idx] * v[idx]
        sm_u[tid] = u[idx] * u[idx]
        sm_v[tid] = v[idx] * v[idx]
    else:
        sm_dot[tid] = 0.0
        sm_u[tid] = 0.0
        sm_v[tid] = 0.0

    cuda.syncthreads()

    # Step 2: reduction inside block
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            sm_dot[tid] += sm_dot[tid + stride]
            sm_u[tid] += sm_u[tid + stride]
            sm_v[tid] += sm_v[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # Step 3: one atomic per block
    if tid == 0:
        cuda.atomic.add(result, 0, sm_dot[0])
        cuda.atomic.add(result, 1, sm_u[0])
        cuda.atomic.add(result, 2, sm_v[0])

In [9]:
result = np.zeros(3, dtype=np.float32)  # [dot, u_norm, v_norm]

u_d = cuda.to_device(u)
v_d = cuda.to_device(v)
result_d = cuda.to_device(result)

cos_sim_kernel_sm[grid_dim, block_dim](u_d, v_d, result_d)
cuda.synchronize()
result = result_d.copy_to_host()

dot, u_norm, v_norm = result
cos_sim = dot / (np.sqrt(u_norm) * np.sqrt(v_norm))

print(f"Cosine similarity: {cos_sim}")

Cosine similarity: 0.7500564455986023


### PyOpenCL version

In [10]:
# Step 1: OpenCL setup
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]

context = cl.Context([device])
queue = cl.CommandQueue(context)

# Step 2: buffers (GPU)
mf = cl.mem_flags

d_u = cl.Buffer(context, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=u)
d_v = cl.Buffer(context, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=v)

# output arrays (per-element contributions)
dot_partial = np.zeros(N, dtype=np.float32)
u_partial   = np.zeros(N, dtype=np.float32)
v_partial   = np.zeros(N, dtype=np.float32)

d_dot = cl.Buffer(context, mf.WRITE_ONLY, dot_partial.nbytes)
d_uout = cl.Buffer(context, mf.WRITE_ONLY, u_partial.nbytes)
d_vout = cl.Buffer(context, mf.WRITE_ONLY, v_partial.nbytes)

# shapes:
# dot_partial, u_partial, v_partial: (N,)

# Step 3: kernel
kernel_code = """
__kernel void cosine_kernel(
    __global const float *u,
    __global const float *v,
    __global float *dot_out,
    __global float *u_out,
    __global float *v_out,
    const int N)
{
    int i = get_global_id(0);

    if (i < N) {
        float ui = u[i];
        float vi = v[i];

        dot_out[i] = ui * vi;
        u_out[i]   = ui * ui;
        v_out[i]   = vi * vi;
    }
}
"""

program = cl.Program(context, kernel_code).build()

# Step 4: launch kernel
global_size = (N,)
local_size = None  # let OpenCL decide

program.cosine_kernel(
    queue,
    global_size,
    local_size,
    d_u, d_v,
    d_dot, d_uout, d_vout,
    np.int32(N)
)

# Step 5: copy back
cl.enqueue_copy(queue, dot_partial, d_dot)
cl.enqueue_copy(queue, u_partial, d_uout)
cl.enqueue_copy(queue, v_partial, d_vout)

# Step 6: final reduction (CPU)
dot = np.sum(dot_partial)
u_norm = np.sum(u_partial)
v_norm = np.sum(v_partial)

cos_sim = dot / (np.sqrt(u_norm) * np.sqrt(v_norm))

print("Cosine similarity:", cos_sim)

Cosine similarity: 0.75005776
